# ESSOS Fieldlines And Particles

## What you will learn
How fieldline and particle diagnostics become validation gates for coil and reactor designs.

## Codes used
ESSOS in research mode; synthetic fieldline/orbit fallback by default.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`06_fieldlines.png`, `06_particle_orbit.png`, and generated GIFs.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
t = np.linspace(0, 18*np.pi, 1400)
r = 1 + 0.18*np.cos(0.22*t)
fieldline = pd.DataFrame({"x": r*np.cos(t/4), "y": r*np.sin(t/4), "z": 0.16*np.sin(t)})
fieldline.head()

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
fig = plt.figure(figsize=(5.8, 4.2))
ax = fig.add_subplot(111, projection="3d")
ax.plot(fieldline.x, fieldline.y, fieldline.z, color="#0f766e", lw=1)
ax.set_axis_off()
ax.set_title("Cached fieldline diagnostic")
savefig(fig, "06_fieldlines.png")
plt.show()
print("Caption: fieldline plots diagnose magnetic topology but do not prove particle confinement.")

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
phi = np.linspace(0, 2*np.pi, 400)
fig, ax = plt.subplots(figsize=(5.6, 4.1))
ax.plot(np.cos(phi), np.sin(phi), lw=2, label="confined orbit")
ax.plot(0.75*np.cos(phi) + 0.25*phi/np.pi, 0.55*np.sin(phi), lw=2, label="loss cartoon")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("Particle orbit cartoon")
caption(ax, "Fast-particle losses are reactor gates even when fieldlines look clean.")
savefig(fig, "06_particle_orbit.png")
plt.show()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
checks = pd.DataFrame({
    "diagnostic": ["fieldline", "guiding-center orbit", "full orbit", "wall hit map"],
    "cost": ["low", "medium", "higher", "postprocess"],
    "design_use": ["topology", "fast-ion screen", "validation", "engineering risk"],
})
checks

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    import essos
    print("ESSOS import OK; research path: run a tiny fieldline tracing example.")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.